In [0]:
%pip install --upgrade replicate

In [0]:
dbutils.library.restartPython()

In [0]:
import base64
import replicate
import os
from PIL import Image
from io import BytesIO
from config import DeployConfig

In [0]:
dbutils.widgets.text("config_path", "./config/env_variables.yml")
config_path = dbutils.widgets.get("config_path")
cfg = DeployConfig.from_yaml(config_path)

In [0]:
image_table = getattr(cfg, f"image_table")
brand_table = getattr(cfg, f"brand_table")

In [0]:
os.environ['REPLICATE_API_TOKEN'] = dbutils.secrets.get("jssandom-scope", "replicate-key")

In [0]:
def resize_to_512(image_bytes):
    """
    Resize an image so the longest side is 512px, keeping aspect ratio.
    Returns the resized image as PNG bytes.
    """
    # Open image from bytes
    img = Image.open(BytesIO(image_bytes))

    # Resize while maintaining aspect ratio
    img.thumbnail((512, 512))  # modifies in-place

    # Convert back to bytes
    buf = BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

In [0]:
pet_image = spark.sql(f'select content from {image_table.path} where id=28').collect()[0]['content']
pet_image = resize_to_512(pet_image)
img = Image.open(BytesIO(pet_image))

img

In [0]:
img.save("pet_image.png")

In [0]:
brand_image = spark.sql(f'select content from {brand_table.path} where version=1').collect()[0]['content']
brand_image = resize_to_512(brand_image)
img = Image.open(BytesIO(brand_image))

img

In [0]:
img.save("brand_image.png")

In [0]:
output = replicate.run(
    "flux-kontext-apps/multi-image-kontext-max",
    input={
        "seed": 42,
        "prompt": "Create a product advertisement for pet food with the images provided. Show the pet from pet image interacting naturally with the bag of pet food found in the petfood brand image. Make it as realistic as possible. Keep the pet in the image's original environment and try and incorporate the background. Do not add any text to the ad except for the text on the bag of pet food.",
        "aspect_ratio": "1:1",
        "input_image_1": open("pet_image.png", "rb"),
        "input_image_2": open("brand_image.png", "rb"),
        "output_format": "png",
        "safety_tolerance": 2
    }
)

# To write the file to disk:
with open("gen_ad_image.png", "wb") as file:
    file.write(output.read())

In [0]:
img = Image.open("gen_ad_image.png")
img.show()